# 04 - Validaciones y Exploración

Objetivo:
- Validar nulos en columnas esenciales de la OBT.
- Validar rangos lógicos (distancias, duraciones, montos).
- Validar coherencia temporal (pickup vs dropoff).
- Revisar duplicados por clave natural (`trip_nk`).
- Comparar conteos RAW vs OBT por servicio/año/mes.
- Construir la matriz de cobertura desde `RAW.LOAD_AUDIT`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import datetime
import glob

In [2]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = ['net.snowflake.spark.snowflake.DefaultSource', 'snowflake.DefaultSource']
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('04_validaciones_y_exploracion')
assert_snowflake_connector(app)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

services = [item.strip() for item in os.environ.get('SERVICES', 'yellow,green').split(',') if item.strip()]
start_year = int(os.environ.get('START_YEAR', '2015'))
end_year = int(os.environ.get('END_YEAR', '2025'))
months = [int(item.strip()) for item in os.environ.get('MONTHS', '1,2,3,4,5,6,7,8,9,10,11,12').split(',') if item.strip()]

log_step(f'Notebook 04 ready: services={services}, years={start_year}-{end_year}, months={months}')

[2026-04-05 23:20:23Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-05 23:20:26Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-05 23:20:26Z] Notebook 04 ready: services=['yellow', 'green'], years=2015-2015, months=[2]


## 1. Cargar OBT desde Snowflake

In [3]:
obt_df = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'ANALYTICS.OBT_TRIPS')
    .load()
)

total_obt_rows = obt_df.count()
log_step(f'Loaded ANALYTICS.OBT_TRIPS rows={total_obt_rows}')
obt_df.printSchema()

[2026-04-05 23:24:34Z] Loaded ANALYTICS.OBT_TRIPS rows=25153846
root
 |-- OBT_TRIP_SK: string (nullable = true)
 |-- TRIP_NK: string (nullable = true)
 |-- TAXI_TYPE: string (nullable = false)
 |-- SERVICE_TYPE: string (nullable = false)
 |-- SOURCE_SERVICE: string (nullable = false)
 |-- PICKUP_DATETIME: timestamp (nullable = true)
 |-- DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PICKUP_DATE: date (nullable = true)
 |-- DROPOFF_DATE: date (nullable = true)
 |-- DROPOFF_HOUR: decimal(38,0) (nullable = true)
 |-- DAY_OF_WEEK: string (nullable = true)
 |-- MONTH: decimal(38,0) (nullable = true)
 |-- YEAR: decimal(38,0) (nullable = true)
 |-- TRIP_DATE: date (nullable = true)
 |-- TRIP_YEAR: decimal(38,0) (nullable = true)
 |-- TRIP_MONTH: decimal(38,0) (nullable = true)
 |-- TRIP_DAY: decimal(38,0) (nullable = true)
 |-- TRIP_DAY_OF_WEEK: string (nullable = true)
 |-- PICKUP_DATETIME_UTC: timestamp (nullable = true)
 |-- DROPOFF_DATETIME_UTC: timestamp (nullable = true)
 |-- PICKU

## 2. Validación de nulos en columnas esenciales

In [ ]:
essential_cols = [
    'trip_nk', 'pickup_datetime_utc', 'dropoff_datetime_utc',
    'PULocationID', 'DOLocationID', 'taxi_type', 'service_type',
    'total_amount', 'trip_duration_minutes'
]

total_rows = obt_df.count()

null_summary = obt_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in essential_cols
])

print(f'Null counts (out of {total_rows} rows):')
null_summary.show(truncate=False)

Null counts (out of 25153846 rows):


## 3. Validación de rangos lógicos

In [ ]:
range_checks = {
    'negative_trip_distance': obt_df.filter(F.col('trip_distance_miles') < 0).count(),
    'negative_trip_duration': obt_df.filter(F.col('trip_duration_minutes') < 0).count(),
    'negative_total_amount': obt_df.filter(F.col('total_amount') < 0).count(),
    'negative_fare_amount': obt_df.filter(F.col('fare_amount') < 0).count(),
    'dropoff_before_pickup': obt_df.filter(F.col('dropoff_datetime_utc') < F.col('pickup_datetime_utc')).count(),
    'zero_distance_positive_fare': obt_df.filter(
        (F.col('trip_distance_miles') == 0) & (F.col('fare_amount') > 0)
    ).count(),
    'speed_over_100mph': obt_df.filter(F.col('avg_speed_mph') > 100).count(),
    'duration_over_24h': obt_df.filter(F.col('trip_duration_minutes') > 1440).count(),
}

print('Range validation results:')
for check, count in range_checks.items():
    status = 'PASS' if count == 0 else f'WARN ({count} rows)'
    print(f'  {check}: {status}')

## 4. Detección de duplicados por clave natural

In [ ]:
dup_df = (
    obt_df.groupBy('trip_nk')
    .count()
    .filter(F.col('count') > 1)
)

dup_count = dup_df.count()
print(f'Duplicate trip_nk groups: {dup_count}')

if dup_count > 0:
    print('Sample duplicates:')
    dup_df.orderBy(F.col('count').desc()).show(10, truncate=False)
else:
    print('No duplicates found - idempotency OK')

## 5. Comparación conteos RAW vs OBT por servicio/año/mes

In [ ]:
# Conteos RAW: union de yellow y green
raw_yellow = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.YELLOW_TRIPS')
    .load()
    .groupBy('service_type', 'source_year', 'source_month')
    .agg(F.count('*').alias('raw_count'))
)

raw_green = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.GREEN_TRIPS')
    .load()
    .groupBy('service_type', 'source_year', 'source_month')
    .agg(F.count('*').alias('raw_count'))
)

raw_counts = raw_yellow.unionByName(raw_green)

# Conteos OBT
obt_counts = (
    obt_df.groupBy(
        F.col('taxi_type').alias('service_type'),
        F.col('source_year'),
        F.col('source_month')
    )
    .agg(F.count('*').alias('obt_count'))
)

# Comparacion
comparison = (
    raw_counts.join(
        obt_counts,
        on=['service_type', 'source_year', 'source_month'],
        how='full'
    )
    .fillna(0, subset=['raw_count', 'obt_count'])
    .withColumn('delta', F.col('raw_count') - F.col('obt_count'))
    .withColumn('pct_retained', 
        F.round(F.col('obt_count') / F.when(F.col('raw_count') > 0, F.col('raw_count')).otherwise(F.lit(None)) * 100, 2)
    )
    .orderBy('service_type', 'source_year', 'source_month')
)

print('RAW vs OBT row comparison:')
comparison.show(300, truncate=False)

## 6. Matriz de cobertura desde LOAD_AUDIT

In [ ]:
audit_df = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.LOAD_AUDIT')
    .load()
)

print('Audit log rows:', audit_df.count())
audit_df.groupBy('status').count().show()

# Matriz de cobertura: servicio x año-mes
coverage_pdf = audit_df.select('service', 'year', 'month', 'status').toPandas()
coverage_pdf['year_month'] = coverage_pdf['year'].astype(str) + '-' + coverage_pdf['month'].astype(str).str.zfill(2)

coverage_matrix = coverage_pdf.pivot_table(
    index='year_month',
    columns='service',
    values='status',
    aggfunc='last'
)

print('Coverage matrix (2015-2025):')
coverage_matrix

## 7. Resumen de conteos por servicio y exploración rápida

In [ ]:
# Conteos por servicio
obt_df.groupBy('taxi_type').count().orderBy('taxi_type').show()

# Estadísticas descriptivas rápidas
obt_df.select(
    F.count('*').alias('total_rows'),
    F.countDistinct('trip_nk').alias('distinct_trip_nk'),
    F.round(F.avg('trip_distance_miles'), 2).alias('avg_distance_mi'),
    F.round(F.avg('trip_duration_minutes'), 2).alias('avg_duration_min'),
    F.round(F.avg('total_amount'), 2).alias('avg_total_amount'),
    F.round(F.avg('tip_pct'), 2).alias('avg_tip_pct'),
    F.round(F.avg('avg_speed_mph'), 2).alias('avg_speed_mph'),
    F.sum(F.when(F.col('pu_zone').isNull(), 1).otherwise(0)).alias('null_pu_zone'),
    F.sum(F.when(F.col('do_zone').isNull(), 1).otherwise(0)).alias('null_do_zone')
).show(truncate=False)

# Snapshot de la OBT
obt_df.select(
    'trip_nk', 'taxi_type', 'trip_date', 'pickup_hour', 'pickup_time_band',
    'pu_borough', 'pu_zone', 'do_borough', 'do_zone',
    'distance_bucket', 'trip_distance_miles', 'trip_duration_minutes', 'avg_speed_mph',
    'payment_type_desc', 'total_amount', 'tip_pct', 'route_key'
).show(20, truncate=False)